In [180]:
import json
import tiktoken

In [181]:
infra_path = "../../infra/"
json_path = f"{infra_path}json/"
cuad_json = f"{infra_path}CUAD_v1/CUAD_v1.json"
cuad_json, json_path

('../../infra/CUAD_v1/CUAD_v1.json', '../../infra/json/')

In [182]:
with open(cuad_json, "r") as f:
    cuad_data = json.load(f)

In [183]:
len(cuad_data['data'])

510

In [184]:
from pydantic import BaseModel, Field
from typing import List, Optional
from pathlib import Path
import re

class Node(BaseModel):
    id: str
    text: str
    relationsCount: int = 0

class Edge(BaseModel):
    source: str
    target: str
    type: str
    score: Optional[float] = None
    ref_label: Optional[str] = None
    ref_value: Optional[str] = None

class Graph(BaseModel):
    nodes: List[Node]
    edges: List[Edge]

class Config:
  CUAD_PDF_DIR = Path("../infra/CUAD_v1/full_contract_pdf")
  CUAD_DOC_DIR = Path("../infra/CUAD_v1/full_contract_docx")
  EDITED_PDF_DIR = Path("../infra/edited_pdfs")
  EDITED_DOC_DIR = Path("../infra/edited_docs")
  REFERENCE_PATTERNS = [
    ("section", re.compile(r'\b[Ss]ection\s+(\d+(?:\.\d+)*)\b')),
    ("article", re.compile(r'\b[Aa]rticle\s+(\d+(?:\.\d+)*)\b')),
    ("schedule", re.compile(r'\b[Ss]chedule\s+([A-Za-z]|\d+(?:\.\d+)*)\b')),
    ("annex",    re.compile(r'\b[Aa]nnex\s+([A-Za-z]|\d+(?:\.\d+)*)\b')),
    ("appendix", re.compile(r'\b[Aa]ppendix\s+([A-Za-z]|\d+(?:\.\d+)*)\b')),
  ]
    
  TYPE_PRIORITY = {
    "precedence": 5,
    "scope": 4,
    "deontic": 3,
    "numeric": 2,
    "definition": 2,
    "other": 1
  }

#### Extract context

In [185]:
new_json_only_context = []

COST_INPUT_PER_MILLION = 0.40  # millon tokens
COST_OUTPUT_PER_MILLION = 1.60  # millon tokens
cost_input_model_per_token = (COST_INPUT_PER_MILLION / 1_000_000)
cost_output_model_per_token = (COST_OUTPUT_PER_MILLION / 1_000_000)
enc = tiktoken.encoding_for_model("gpt-4.1-mini")

index = 1
for item in cuad_data['data']:
    title = item['title']
    context = item['paragraphs'][0]['context']
    number_tokens = len(enc.encode(context))

    new_json_only_context.append({
        "id": f"CUAD_{index}",
        "title": title,
        "context": context,
        'total_number_tokens': number_tokens,
        'cost_document_input': ((number_tokens / 1_000_000) * COST_INPUT_PER_MILLION),
        'cost_document_output': ((number_tokens / 1_000_000) * COST_OUTPUT_PER_MILLION)
    })

    index += 1

In [186]:
document_output_path = f"{json_path}rerank/context.json" 

with open(document_output_path, "w") as f:
    json.dump(new_json_only_context, f, indent=4)

In [187]:
INPUT_PATH = f"{json_path}/rerank/context.json"
OUTPUT_PATH = f"{json_path}/rerank/paragraphs.json"

MAX_DOCS = 1
threshold_tokens = 12000

def split_into_paragraphs(text: str):
    text = text.replace("\r\n", "\n").strip()
    parts = re.split(r"\n\s*\n+", text)

    out = []
    for part in parts:
        clean = " ".join(part.split())
        if clean:
            out.append(clean)

    if out:
        return out

    clean = " ".join(text.split())
    return [clean] if clean else []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

candidate_docs = docs

print(len(candidate_docs), "\n")

selected = []
for i, doc in enumerate(candidate_docs, start=1):
    context = doc.get("context", "")
    total_tokens_doc = len(enc.encode(context))

    if total_tokens_doc >= threshold_tokens:
        selected.append(doc)

selected = sorted(selected, key=lambda d: len(enc.encode(d.get("context", ""))), reverse=False)

print(len(selected), "\n")

stage1 = []

for doc in selected[:MAX_DOCS]:
    paragraphs = []

    for i, p_text in enumerate(split_into_paragraphs(doc.get("context", "")), start=1):
        n_tokens = len(enc.encode(p_text))

        paragraphs.append(
            {
                "idx": f"IDX{i}",
                "text": p_text,
                "number_tokens": n_tokens,
                "cost_document_input": (n_tokens / 1_000_000) * cost_input_model_per_token,
                "cost_document_output": (n_tokens / 1_000_000) * cost_output_model_per_token,
            }
        )

    stage1.append(
        {
            "doc_id": doc.get("id"),
            "num_paragraphs": len(paragraphs),
            "total_number_tokens": sum(p["number_tokens"] for p in paragraphs),
            "cost_document_input": sum(p["cost_document_input"] for p in paragraphs),
            "paragraphs": paragraphs,
        }
    )

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(stage1, f, indent=2, ensure_ascii=False)

510 

148 



#### Create Graph

In [188]:
from sentence_transformers import SentenceTransformer, util

def generate_graph_data(paragraphs_data: list) -> Graph:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    nodes: List[Node] = []
    edges: List[Edge] = []
    
    for p in paragraphs_data:
        node = Node(
            id=str(p.get("idx")),
            text=p.get("text", "").strip(),
            relationsCount=0,
        )
        nodes.append(node)
    
    for i in range(len(nodes)):
        current_text = nodes[i].text
        for ref_type, pattern in Config.REFERENCE_PATTERNS:
            matches = pattern.finditer(current_text)
            for match in matches:
                ref_id = match.group(1)
                for target_node in nodes:
                    if target_node.id != nodes[i].id and target_node.text.startswith(ref_id):
                        edges.append(Edge(
                            source=nodes[i].id, 
                            target=target_node.id, 
                            type="reference",
                            ref_label=ref_type,
                            ref_value=ref_id
                        ))

    if nodes:
        embeddings = model.encode([n.text for n in nodes], convert_to_tensor=True)
        cosine_scores = util.cos_sim(embeddings, embeddings)

        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                score = float(cosine_scores[i][j])
                if score > 0.8:
                    edges.append(Edge(
                        source=nodes[i].id, 
                        target=nodes[j].id, 
                        type="semantic_similarity", 
                        score=score
                    ))

    relations_map = {}
    for edge in edges:
        relations_map[edge.source] = relations_map.get(edge.source, 0) + 1
        relations_map[edge.target] = relations_map.get(edge.target, 0) + 1
    
    for node in nodes:
        node.relationsCount = relations_map.get(node.id, 0)

    return Graph(nodes=nodes, edges=edges)

In [189]:
graph = generate_graph_data(stage1[0]["paragraphs"])
graph

Graph(nodes=[Node(id='IDX1', text='1 EXHIBIT 10.17 [E.PIPHANY Logo]', relationsCount=0), Node(id='IDX2', text='OUTSOURCING AGREEMENT', relationsCount=0), Node(id='IDX3', text='This ASP and Outsourcing Agreement ("Agreement") is entered into as of this 31 day of July, 2000 ("Effective Date") by and between E.PIPHANY, INC., a Delaware corporation ("E.piphany"), whose principal place of business 1900 South Norfolk Street, Suite 310, San Mateo, California 94403 and HIGH SPEED NET SOLUTIONS, INC. ("HSNS"), whose principal place of business is 434 Fayetteville Street, St. Suite 2120, Raleigh, NC 27601.', relationsCount=0), Node(id='IDX4', text='1. LICENSE', relationsCount=0), Node(id='IDX5', text='1.1 OUTSOURCING LICENSE. Subject to the terms of this Agreement and Scope of Use and only within the Market and Territory, E.piphany grants HSNS a nonexclusive, nontransferable, non-sublicensable right to (i) use and combine the Application with the Outsourcing Application and other software produc

#### "Query-like object”

Reranker Concept for Graph-Based Contradiction Generation

In this scenario there is no external query. The "query-like object" is the target node where the contradiction will be inserted.

Instead of asking:

> "What paragraphs are most relevant to answer a user question?"

The system asks:

> "Given node ``Idx5``, which nodes in the graph are the most relevant to condition, restrict, or contextualize a contradiction that will be inserted into ``Idx5``?"

This changes the logic of reranking.

The reranker in this pipeline is better described as a:

- node-to-node relevance ranker
or
- context selector for contradiction generation

Inputs:
- target node (``v_target``)
- candidate neighbors N(``v_target``) or nodes within 1–2 hops
- the local subgraph

Output:
- ranking of nodes most relevant to the target node

No textual query is used. The target node acts as the semantic anchor.

--------------------------------------------------

Pipeline

1. The document is divided into paragraphs.

2. A graph is constructed where:
   - nodes = paragraphs
   - edges = references or semantic similarity

3. A target node ``v_target`` is selected.

4. Candidate nodes related to ``v_target`` are retrieved.
   For example:
   - direct neighbors
   - semantic neighbors
   - reference links
   Example size: top-5 candidates.

5. A reranker orders the candidates based on relevance to ``v_target``.

6. The system selects the top-3 nodes.

7. The contradiction generation module receives:
   - target paragraph
   - top-3 contextual nodes

8. A contradiction is generated for the target node.

9. The modified paragraph is inserted back into the document.

10. The resulting document becomes part of the contradiction dataset.

--------------------------------------------------

Meaning of "Relevance"

Relevance here does not mean answering a question. Instead it means identifying nodes that influence the logical interpretation of the target node.

Relevant nodes may include:

- paragraphs that define terms used in the target node
- paragraphs that introduce exceptions
- paragraphs that establish precedence rules
- paragraphs that reference the target node
- paragraphs that depend logically on the target node

--------------------------------------------------

Retrieve vs Rerank

Retrieve step:
The goal is recall. We gather a reasonably large set of potentially useful nodes.

Example sources:
- graph neighbors
- semantic similarity edges
- reference edges

Rerank step:
The goal is precision. The system orders nodes according to how strongly they interact with the target node.

--------------------------------------------------

Signals for Node Relevance

Possible signals include:

1. Explicit references
If a node references another node, this relationship is highly informative.

Example patterns:
- "subject to Section 4.2"
- "as defined in Section 1"
- "except as provided in Clause 9"

2. Semantic similarity
Embedding similarity between nodes.

3. Graph structure
Number of edges and connectivity.

4. Graph distance
Nodes closer in the graph may be more relevant.

5. Legal role of the paragraph
Some paragraphs are structurally more important.

Examples:
- definitions
- obligations
- exceptions
- scope restrictions

6. Shared entities or terminology
Overlap in legal entities, dates, numbers, or defined terms.

--------------------------------------------------

Conceptual Ranking Function

$$
\text{score}(\text{candidate} \mid \text{target}) = \alpha \cdot \text{semantic\_similarity}(\text{target}, \text{candidate}) + \beta \cdot \text{reference\_strength}(\text{target}, \text{candidate}) + \gamma \cdot \text{legal\_interaction}(\text{target}, \text{candidate}) + \delta \cdot \text{graph\_distance}(\text{target}, \text{candidate}) + \varepsilon \cdot \text{entity\_overlap}(\text{target}, \text{candidate})
$$

--------------------------------------------------

Example Scenario

Target node: ``Idx5``

Candidate nodes retrieved:
Idx1, Idx2, Idx3, Idx4, Idx6

After reranking:

1. Idx3
2. Idx2
3. Idx6
4. Idx4
5. Idx1

Top-3 context nodes:
Idx3, Idx2, Idx6

The contradiction generator uses:

- text of ``Idx5``
- context from Idx3, Idx2, Idx6

to generate a coherent but contradictory modification of ``Idx5``.

--------------------------------------------------

Module Structure

- Module 1: Candidate Retrieval
   - Input: target node
   - Output: set of candidate nodes

- Module 2: Candidate Reranking
   - Input: target node + candidate nodes
   - Output: ranked nodes

- Module 3: Contradiction Generation
   - Input: target node + top context nodes
   - Output: contradictory paragraph

- Module 4: Document Reconstruction 
   - Insert modified paragraph into the original document.

In [190]:
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
import re

rerank_model = SentenceTransformer("all-MiniLM-L6-v2")

In [191]:
def get_node_by_id(graph: Graph, node_id: str) -> Optional[Node]:
    for node in graph.nodes:
        if node.id == node_id:
            return node
    return None

# It allows you to quickly answer questions like:
# Which nodes does IDX5 point to?
# Which nodes point to IDX5?
# Who are its neighbors?

def build_adjacency(graph: Graph):
    outgoing = defaultdict(list) # the edges that come out of each node
    incoming = defaultdict(list) # the edges that enter each node

    for edge in graph.edges:
        outgoing[edge.source].append(edge)
        incoming[edge.target].append(edge)

    return outgoing, incoming


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"\w+", text.lower()))

In [192]:
def classify_legal_type(text: str) -> str:
    t = text.lower()

    if any(x in t for x in ["notwithstanding", "except as", "subject to", "in the event of conflict"]):
        return "precedence"

    if any(x in t for x in ["only", "solely", "limited to", "for purposes of", "applies to"]):
        return "scope"

    if any(x in t for x in ["shall", "must", "will", "is required to", "obligated to"]):
        return "deontic"

    if re.search(r"\b\d+\b", t) or any(x in t for x in ["$", "%", "days", "months", "years"]):
        return "numeric"

    if any(x in t for x in [" means ", "defined as", "definition", "refers to"]):
        return "definition"

    return "other"

In [193]:
def lexical_overlap_score(text_a: str, text_b: str) -> float:
    a = tokenize(text_a)
    b = tokenize(text_b)

    if not a:
        return 0.0

    return len(a & b) / len(a)

In [194]:
def retrieve_candidate_nodes(graph: Graph, target_id: str, max_candidates: int = 5) -> list[str]:
    outgoing, incoming = build_adjacency(graph)

    candidate_scores = defaultdict(float)

    for edge in outgoing.get(target_id, []):
        if edge.target != target_id:
            if edge.type == "reference":
                candidate_scores[edge.target] += 3.0
            elif edge.type == "semantic_similarity":
                candidate_scores[edge.target] += edge.score if edge.score is not None else 1.0
            else:
                candidate_scores[edge.target] += 1.0

    for edge in incoming.get(target_id, []):
        if edge.source != target_id:
            if edge.type == "reference":
                candidate_scores[edge.source] += 3.0
            elif edge.type == "semantic_similarity":
                candidate_scores[edge.source] += edge.score if edge.score is not None else 1.0
            else:
                candidate_scores[edge.source] += 1.0

    ranked_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [node_id for node_id, _ in ranked_candidates[:max_candidates]]

In [195]:
def rank_nodes_for_target(graph: Graph, target_id: str, candidate_ids: list[str], top_k: int = 3):
    target_node = get_node_by_id(graph, target_id)
    if target_node is None:
        raise ValueError(f"Target node {target_id} not found")

    outgoing, incoming = build_adjacency(graph)

    target_text = target_node.text
    target_emb = rerank_model.encode(target_text, convert_to_tensor=True)

    max_rel = max((node.relationsCount for node in graph.nodes), default=1)

    ranked = []

    for candidate_id in candidate_ids:
        candidate_node = get_node_by_id(graph, candidate_id)
        if candidate_node is None or candidate_node.id == target_id:
            continue

        candidate_text = candidate_node.text
        candidate_emb = rerank_model.encode(candidate_text, convert_to_tensor=True)

        semantic_sim = float(util.cos_sim(target_emb, candidate_emb)[0][0])

        overlap = lexical_overlap_score(target_text, candidate_text)

        relation_score = candidate_node.relationsCount / max_rel if max_rel > 0 else 0.0

        direct_reference_bonus = 0.0
        semantic_edge_bonus = 0.0

        for edge in outgoing.get(target_id, []):
            if edge.target == candidate_id:
                if edge.type == "reference":
                    direct_reference_bonus += 1.0
                elif edge.type == "semantic_similarity":
                    semantic_edge_bonus += edge.score if edge.score is not None else 0.5

        for edge in incoming.get(target_id, []):
            if edge.source == candidate_id:
                if edge.type == "reference":
                    direct_reference_bonus += 1.0
                elif edge.type == "semantic_similarity":
                    semantic_edge_bonus += edge.score if edge.score is not None else 0.5

        legal_type = classify_legal_type(candidate_text)
        priority_score = Config.TYPE_PRIORITY.get(legal_type, 1) / 5.0

        final_score = (
            0.35 * semantic_sim +
            0.20 * overlap +
            0.20 * direct_reference_bonus +
            0.10 * semantic_edge_bonus +
            0.10 * relation_score +
            0.05 * priority_score
        )

        ranked.append({
            "target_id": target_id,
            "candidate_id": candidate_id,
            "legal_type": legal_type,
            "semantic_sim": round(semantic_sim, 4),
            "lexical_overlap": round(overlap, 4),
            "direct_reference_bonus": round(direct_reference_bonus, 4),
            "semantic_edge_bonus": round(semantic_edge_bonus, 4),
            "relation_score": round(relation_score, 4),
            "priority_score": round(priority_score, 4),
            "final_score": round(final_score, 4),
            "text": candidate_text,
        })

    ranked = sorted(ranked, key=lambda x: x["final_score"], reverse=True)
    return ranked[:top_k]

In [196]:
target_id = "IDX5"

candidate_ids = retrieve_candidate_nodes(graph, target_id=target_id, max_candidates=5)
print("Candidates:", candidate_ids)

top_context = rank_nodes_for_target(
    graph=graph,
    target_id=target_id,
    candidate_ids=candidate_ids,
    top_k=3
)

for item in top_context:
    print(item["candidate_id"], item["final_score"], item["legal_type"])
    print(item["text"][:300])
    print("-" * 80)

Candidates: ['IDX8']
IDX8 0.4988 precedence
1.4 LICENSE RESTRICTIONS. Except as expressly provided herein, HSNS shall not (i) rent, lease, loan, sell or otherwise distribute the Application, or any modification thereto, in whole or in part; (ii) cause or permit reverse engineering, reverse compilation, unauthorized access or assembly of all o
--------------------------------------------------------------------------------


```python
stage1[0]["paragraphs"] -> generate_graph_data(...) -> graph
graph + target_id -> retrieve_candidate_nodes(...)
target_id + candidate_ids -> rank_nodes_for_target(...)
top-3 -> contradiction generator
```